# Bayesian Logistic Regression - Bank Marketing
Predict bank customer attrition using classification models.

## Project Overview
Binary classification to predict credit card customer churn (attrition) from the BankChurners dataset. Despite the folder name, we use standard and boosting classifiers.

## Learning Objectives
- Predict customer churn in banking
- Handle categorical features (education, marital status, income)
- Understand churn drivers for retention strategies

## Problem Statement
Given a credit card customer's demographics and account details, predict whether they will attrit (leave the bank).

## Why This Project Matters
Customer attrition directly impacts bank revenue. Identifying at-risk customers enables proactive retention efforts.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Local: BankChurners.csv |
| **Target** | Attrition_Flag (Existing/Attrited Customer) |
| **Rows** | ~10,127 |
| **Features** | Demographics, card details, transaction history |

## Dataset Source & License
Kaggle BankChurners dataset. Public domain for educational use.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
TARGET = 'Attrition_Flag'

## Dataset Loading

In [4]:
csv_path = os.path.join(SAVE_DIR, 'BankChurners.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Dataset not found at {csv_path}')
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
df.head()

Shape: (10127, 21)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nTarget distribution:')
print(df[TARGET].value_counts())

Missing: 0
Duplicates: 0

Target distribution:
Attrition_Flag
Existing Customer    8500
Attrited Customer    1627
Name: count, dtype: int64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#2ecc71','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Attrition Distribution')

if 'Customer_Age' in df.columns:
    for label in df[TARGET].unique():
        axes[0,1].hist(df[df[TARGET]==label]['Customer_Age'], bins=30, alpha=0.5, label=label[:10])
    axes[0,1].legend()
    axes[0,1].set_title('Age by Attrition')

if 'Total_Trans_Ct' in df.columns:
    for label in df[TARGET].unique():
        axes[1,0].hist(df[df[TARGET]==label]['Total_Trans_Ct'], bins=30, alpha=0.5, label=label[:10])
    axes[1,0].legend()
    axes[1,0].set_title('Transaction Count by Attrition')

if 'Credit_Limit' in df.columns:
    for label in df[TARGET].unique():
        axes[1,1].hist(df[df[TARGET]==label]['Credit_Limit'].clip(upper=20000), bins=30, alpha=0.5, label=label[:10])
    axes[1,1].legend()
    axes[1,1].set_title('Credit Limit by Attrition')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print(f'\nAttrition rate: {(df[TARGET]=="Attrited Customer").mean():.2%}')

Attrition_Flag
Existing Customer    8500
Attrited Customer    1627
Name: count, dtype: int64

Attrition rate: 16.07%


## Train/Test Split

In [8]:
# Drop CLIENTNUM and any naive bayes columns from Kaggle
drop_cols = [c for c in df.columns if 'CLIENTNUM' in c or 'Naive' in c]
df = df.drop(columns=drop_cols, errors='ignore')

# Encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET])
print(f'Classes: {le.classes_}')

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes

df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Classes: ['Attrited Customer' 'Existing Customer']
Train: (8101, 19), Test: (2026, 19)


## Preprocessing
Dropped client ID and naive bayes columns. Encoded categoricals. Filled missing.

## Feature Engineering
Using raw customer and transaction features. Tree models handle interactions.

## Baseline Model

In [9]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')

Baseline LR: Acc=0.8963  F1=0.9398


## LazyPredict Benchmark

In [10]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                         
CatBoostClassifier             0.968           0.929188  0.991205  0.967657   
LGBMClassifier                 0.968           0.926597  0.987601  0.967568   
XGBClassifier                  0.967           0.926004  0.988916  0.966601   
BaggingClassifier              0.955           0.918886  0.958380  0.955173   
RandomForestClassifier         0.962           0.902305  0.986717  0.960930   
DecisionTreeClassifier         0.933           0.877330  0.877330  0.933257   
AdaBoostClassifier             0.941           0.848384  0.968047  0.938301   
ExtraTreesClassifier           0.942           0.838611  0.984212  0.938438   
ExtraTreeClassifier            0.889           0.812359  0.812359  0.891285   
SVC                            0.925           0.810387  0.960401  0.920796   
LinearDiscriminantAnalysis     0.899           0.771

## FLAML AutoML

In [11]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Acc={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb_m = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_m.fit(X_train, y_train)
models['XGBoost'] = xgb_m

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

CatBoost: Acc=0.9753  F1=0.9751
LightGBM: Acc=0.9743  F1=0.9742
XGBoost: Acc=0.9743  F1=0.9743


## Top 3 Model Selection

In [13]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision    Recall
CatBoost  0.975321  0.975132   0.975068  0.975321
XGBoost   0.974334  0.974269   0.974218  0.974334
LightGBM  0.974334  0.974171   0.974094  0.974334

Top 3: ['CatBoost', 'XGBoost', 'LightGBM']


## Final Evaluation of Top 3

In [14]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

CatBoost: Acc=0.9753  F1=0.9751
              precision    recall  f1-score   support

           0       0.94      0.90      0.92       325
           1       0.98      0.99      0.99      1701

    accuracy                           0.98      2026
   macro avg       0.96      0.95      0.95      2026
weighted avg       0.98      0.98      0.98      2026

XGBoost: Acc=0.9743  F1=0.9743
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       325
           1       0.98      0.99      0.98      1701

    accuracy                           0.97      2026
   macro avg       0.95      0.95      0.95      2026
weighted avg       0.97      0.97      0.97      2026

LightGBM: Acc=0.9743  F1=0.9742
              precision    recall  f1-score   support

           0       0.93      0.90      0.92       325
           1       0.98      0.99      0.98      1701

    accuracy                           0.97      2026
   macro avg       0.96      0.95

## Error Analysis

In [15]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

err = (preds != y_test.values).sum()
print(f'Errors: {err}, Error rate: {err/len(y_test):.4f}')

Errors: 50, Error rate: 0.0247


## Interpretation & Business Insight
Total transaction count and revolving balance are top attrition predictors. Customers with declining activity are at highest risk.

## Limitations
- Imbalanced (~16% attrition)
- No temporal sequence of transactions
- Missing customer interaction history

## How to Improve
- Add transaction trend features (month-over-month change)
- Survival analysis for time-to-churn
- Cost-sensitive optimization

## Production Considerations
- Monthly scoring pipeline
- Retention offer triggers above threshold
- A/B test retention interventions

## Common Mistakes
- Including Naive Bayes prediction columns (data leakage!)
- Not dropping CLIENTNUM
- Ignoring class imbalance

## Mini Challenge
1. Create 'activity_decline' feature from transaction changes
2. Try threshold optimization for precision/recall tradeoff
3. Compare with/without Naive Bayes columns

## Final Summary
Predicted bank customer attrition from account and transaction features. Boosting models achieve strong F1 on this moderately imbalanced dataset.

In [16]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "CatBoost": {
    "Accuracy": 0.9753,
    "F1": 0.9751,
    "Precision": 0.9751,
    "Recall": 0.9753
  },
  "XGBoost": {
    "Accuracy": 0.9743,
    "F1": 0.9743,
    "Precision": 0.9742,
    "Recall": 0.9743
  },
  "LightGBM": {
    "Accuracy": 0.9743,
    "F1": 0.9742,
    "Precision": 0.9741,
    "Recall": 0.9743
  },
  "best_model": "CatBoost"
}
